## Brain tumour image classification with CNNs


**Assessment:** PASS/FAIL; complete the assessed tasks and explain your results to a GTA.  
**Choice:** Part 1 (this notebook) **or** Part 2 (NLP, separate activity in the accompanying brief).

### Learning outcomes
1. Load and inspect a three-class image dataset, including class imbalance.
2. Design, compile and train a CNN using TensorFlow/Keras.
3. Compare **three controlled experiments** using validation results.
4. Select a model without using the test set during development.
5. Interpret accuracy, precision, recall, F1-score and a confusion matrix.

**Important:** This is an educational exercise, not a validated medical diagnostic system. The dataset comes from the `BrainTumorDataset.zip`; it is **not bundled** in this notebook.

**Working rule:** Run cells from the top. Change only the parameters explicitly requested. The test set is reserved for the *final* model after selection.

### Workflow and evidence

| Stage | Dataset | Purpose |
|---|---|---|
| 1. Fit the CNN | **Training** | Learn model weights; class weights may be applied here |
| 2. Compare experiments | **Validation** | Compare A, B and C; select the best model by **validation macro-F1** (tie-break: lower validation loss) |
| 3. Evaluate chosen model | **Testing** | One final evaluation; do **not** use these results to revise the model |

**Report these experiments:** A = baseline (no dropout, no class weights); B = dropout only; C = dropout plus class weights. D = An experiment proposed by you. Keep dataset split, architecture other than dropout, random seed, maximum epochs and image resolution fixed.

**Assessment evidence:** class-count table, example images, model summary, three-row experiment table, raw learning curves, final classification report/confusion matrix, and concise explanations.

# Import necessary libraries

In [ ]:
%matplotlib inline
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
import PIL.Image
import tensorflow_datasets as tfds
import pathlib
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

# Set the default figure size for matplotlib
plt.rcParams['figure.figsize'] = (7,7) # Make the figures a bit bigger

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow version:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

# Load data and preprocess it

## Upload the data to drive:
Download the dataset from Brigthspace, and upload it to the content folder in Google Colab.


###Option 1: To avoid having to upload the dataset every time, you can upload the dataset to your google drive, and then connect this virtual machine to your drive, and make a copy of your data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Replace the path /content/drive/MyDrive/Work/2025-CS3002/BrainTumorDataset.zip with your actual path where you copied the dataset in your drive.

In [ ]:
!cp /content/drive/MyDrive/Work/2026-CS3613/BrainTumorDataset.zip /content/

###Option 2: Upload the zip archive in the Files section of your virtual machine, under /content/

###After the data is loaded, let's start processing it.

In [ ]:
# After you have copied the data locally, point the dataset_url to the local path
dataset_url = '/content/BrainTumorDataset.zip'

In [ ]:
import zipfile
import os

# Path to your local zip file
zip_path = dataset_url

# Path where you want to extract
out_path = "/content/"

# Extract
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(out_path)

print("Extraction complete. Files are in:", out_path)


In [ ]:
# Create a 'pathlib.Path' object for the downloaded archive
# Pathlib module offers classes representing filesystem paths with semantics
# appropriate for different operating systems.
# data_dir = pathlib.Path(archive).with_suffix('')

extract_dir = pathlib.Path(zip_path).with_suffix('')
# Now use Pathlib for further work
data_dir = pathlib.Path(extract_dir)

print("Data directory is:", data_dir)

In [ ]:
# Count the number of images in a specific directory
image_count = len(list(data_dir.glob('./Training/glioma/*.jpg')))
print(image_count)

**T1A.** Count the number of images for each class in each split. State which split(s) are balanced. Explain why training imbalance might affect overall accuracy and per-class recall.

In [ ]:
# BEGIN YOUR CODE HERE

# Brain tumour dataset is split into train, validation and test folders. Inside those folders you will
# find additional folders: glioma, pituitary, notumor. You can explore the folders using
# 'Files' tab from the right hand side.
# Tip: use the len() function

# 1a. How many images do we have in training for positive glioma? How many for pituitary? How many for no tumour?


# 1b. How about in the validation set?


# 1c. How about in the testing set?



# END YOUR CODE HERE

In [ ]:
# Create a list of file paths for glioma images
positive_images = list(data_dir.glob('Training/glioma/*'))
# Open and display the first glioma image in the list
PIL.Image.open(str(positive_images[0]))

**T1B.** Display **one validation image per class**. In your answer, explain what resizing to a common image shape accomplishes and one potential limitation of image resizing.

In [ ]:
# BEGIN YOUR CODE HERE

# 2. Display one glioma, one pituitary, and a normal image from the validation dataset


# END YOUR CODE HERE

**T1C.** Explain why unequal training-class frequencies may affect the model’s predictions and evaluation metrics

# Task 2. Define a deep learning model that will learn the differences between glioma and no tumour images

In [ ]:
# Define batch size and image dimensions for training

# BEGIN YOUR CODE HERE
# The batch size is the number of samples processed before the model is updated.
# Choose an appropriate batch size. What are the images resolutions?

# batch_size = TODO


# What is the resolution of the images?
# Specify the size to resize images to after they are read from disk.
# Since the pipeline processes batches of images that must all have the same size, this must be provided.

# img_height = TODO
# img_width = TODO


# END YOUR CODE HERE

In [ ]:
import os
from PIL import Image

def find_min_max_resolutions(folder_path):
    min_res = None
    max_res = None
    min_file = None
    max_file = None

    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')):
                file_path = os.path.join(root, file)
                try:
                    with Image.open(file_path) as img:
                        width, height = img.size
                        resolution = width * height

                        # Update min
                        if min_res is None or resolution < min_res:
                            min_res = resolution
                            min_file = (file_path, (width, height))

                        # Update max
                        if max_res is None or resolution > max_res:
                            max_res = resolution
                            max_file = (file_path, (width, height))

                except Exception as e:
                    print(f"Could not process {file_path}: {e}")

    return min_file, max_file

# Example usage:

folder = "/content/BrainTumorDataset/Training/notumor/"
min_file, max_file = find_min_max_resolutions(folder)

print(f"Smallest image: {min_file[0]} with resolution {min_file[1]}")
print(f"Largest image: {max_file[0]} with resolution {max_file[1]}")


In [ ]:
train_data_dir  = os.path.join(data_dir,'Training')
valid_data_dir = os.path.join(data_dir,'Validation')
test_data_dir = os.path.join(data_dir,'Testing')

In [ ]:
# Create a TensorFlow image dataset from a directory
# BEGIN YOUR CODE HERE
# Use the function tf.keras.utils.image_dataset_from_directory to load
# the training dataset: https://www.tensorflow.org/api_docs/python/tf/keras/utils/image_dataset_from_directory
# 1. First argument is your training directory folder (train_data_dir),
# 2. Do NOT use validation_split since we only want a training set,
# 3. You can set a seed such that when you repeat experiments you get similar results, e.g. seed=123,
# 4. For image size, use the img_height and img_width variables you defined previously,
# 5. For batch size, use the batch_size variable you defined earlier in the code.
# tf.keras.utils.image_dataset_from_directory(
#     directory,
#     seed=None,
#     image_size=(height, width),
#     batch_size=-1,
#     shuffle=False
# )
# Example:




# END YOUR CODE HERE

In [ ]:
# Create a layer to normalise pixel values to the [0, 1] range.
# By default, when you load an image, each pixel value will have a value between 0-255
# but, in neural networks, we need as input normalised values in [0,1] interval.
normalization_layer = tf.keras.layers.Rescaling(1./255)

In [ ]:
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
# Notice the pixel values are now in `[0,1]`.
print(np.min(first_image), np.max(first_image), first_image[0][0])

In [ ]:
# BEGIN YOUR CODE HERE
# Define the number of classes in the classification problem
# How many classes do we have in this dataset?
# num_classes = TODO


# END YOUR CODE HERE

## Task 3 | Four controlled experiments

A: no dropout, no weighting.  B: dropout rate, no weighting.  C: dropout rate plus inverse-frequency class weights.  All other experimental settings remain fixed. D. Your own propose experiment. Each run **builds a new model** and resets the random seed. An `EarlyStopping` callback restores the weights with lowest validation loss.

Calculate training-only class weights using `N / (K × n_class)` and explain which class receives the largest weight and why.


In [ ]:
# Extract training labels
y_train = np.concatenate([y for x, y in train_ds], axis=0)

# Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

# TIP: Search how you can use the class weights during training to improve your results

Helper function for visualisation

In [ ]:
from scipy.ndimage import gaussian_filter1d  # for smoothing

# function to plot loss/accuracy with trend lines
def plot_loss_accuracy(history, smooth_sigma=2):
    # Extract loss and accuracy values
    loss_values = history.history['loss']
    acc_values = history.history['accuracy']

    # Validation (if available)
    val_loss = history.history.get('val_loss')
    val_acc = history.history.get('val_accuracy')

    epochs = np.arange(1, len(loss_values) + 1)

    # --- Plot Training & Validation Loss ---
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss_values, marker='o', linestyle='-', color='b', alpha=0.5, label='Training Loss')
    # plt.plot(epochs, gaussian_filter1d(loss_values, sigma=smooth_sigma), color='b', linewidth=2, label='Trend (Train)')

    if val_loss is not None:
        plt.plot(epochs, val_loss, marker='x', linestyle='--', color='r', alpha=0.5, label='Validation Loss')
        # plt.plot(epochs, gaussian_filter1d(val_loss, sigma=smooth_sigma), color='r', linewidth=2, label='Trend (Val)')

    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Loss Over Epochs')
    plt.grid(True)
    plt.legend()

    # --- Plot Training & Validation Accuracy ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, acc_values, marker='o', linestyle='-', color='g', alpha=0.5, label='Training Accuracy')
    # plt.plot(epochs, gaussian_filter1d(acc_values, sigma=smooth_sigma), color='g', linewidth=2, label='Trend (Train)')

    if val_acc is not None:
        plt.plot(epochs, val_acc, marker='x', linestyle='--', color='orange', alpha=0.5, label='Validation Accuracy')
        # plt.plot(epochs, gaussian_filter1d(val_acc, sigma=smooth_sigma), color='orange', linewidth=2, label='Trend (Val)')

    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Accuracy Over Epochs')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

**T3B.** Train A, B, C and D. Use the provided configuration and logging structure. For each experiment, report validation accuracy, macro-F1, loss, elapsed epochs and number of parameters. **Do not call `test_ds` here.**

In [ ]:
# BEGIN YOUR CODE HERE
# DEFINE MODEL A
# Define a tensorflow model using the tf.keras.Sequential class: https://www.tensorflow.org/api_docs/python/tf/keras/Sequential
# The last layer should be a Dense layer with the number of output neurons num_classes
# Use as a starting point the tf.keras.Sequential model defined for the MNIST problem.
# See Lab-DeepLearning-ImageClassification.
# For the first Conv2D layer, you are not required to specify the input shape. If that
# parameter is not given, tensorflow library will infer the size of the input when
# you fit the model, so it will depend on the size of the dataset.
# Important: change the output of the last Dense layer to match the number of classes for this problem.
# Don't use dropout layers for this version




# compile the model


# END YOUR CODE HERE

In [ ]:
# TRAIN MODEL A
# BEGIN YOUR CODE HERE
# Train the model on the provided dataset for a specified number of epochs
# Modify the network architecture such that you maximise the accuracy.
# Start training using a small number of epochs (for example 5), while you get your code to run.
# Once you are ready you can increase the number of epochs.
# How about if you increase the number of epochs?
# Use as validation_data the val_ds variable you defined previously



# END YOUR CODE HERE

In [ ]:
plot_loss_accuracy(history_A)

In [ ]:
# BEGIN YOUR CODE HERE
# DEFINE MODEL  B, train, plot results


# compile model


# train model


# plot loss/accuracy

# END YOUR CODE HERE

In [ ]:
# BEGIN YOUR CODE HERE
# DEFINE MODEL  C, train, plot results


# compile model


# train model


# plot loss/accuracy

# END YOUR CODE HERE

In [1]:
# BEGIN YOUR CODE HERE
# DEFINE MODEL  D, train, plot results


# compile model


# train model


# plot loss/accuracy

# END YOUR CODE HERE

## Task 4 | Select once, then evaluate on the held-out test set

**Selection rule:** highest **validation macro-F1**; tie-breaker is lower validation loss. Do not inspect or use test results to change this choice.

In [ ]:
# Let's load the testing dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
  test_data_dir,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  shuffle=False   # 🔑 Ensure labels line up with predictions)
)

In [ ]:
# BEGIN YOUR CODE HERE
# What is the loss and accuracy on the Testing dataset?
# Tip: instead of (x_test, y_test) we used in the lab last week, you can use
# directly test_ds which contains both data and labels
# https://www.tensorflow.org/api_docs/python/tf/keras/Model#evaluate
# When you print the output of the evaluate function is run, it will return both
# the loss and accuracy, maybe in a  format like [loss_value, accuracy_value]
# print("Model accuracy on the test set is:", model.evaluate(test_ds))

best_model = TODO

# Evaluate model on test set


# Get true labels and predictions


# Print classification report


# Compute and plot confusion matrix


# END YOUR CODE HERE

In [ ]:
# Try to improve the model as much as you can